# JusticeFlow TZ — Colab walkthrough

Case-delay risk prediction + CP-SAT cause-list optimisation, with a fairness gate and registrar approval.
Runs entirely in the browser (SRS 2.4). Run cells top to bottom.

**Pipeline:** synthetic/TanzLII data → features (region & station held out) → risk model → **fairness gate** → CP-SAT optimiser → **registrar review** → dashboard tables.

## 1. Setup
Upload `justiceflow.zip` (the `justiceflow/` folder zipped) using the file pane on the left, **or** set `REPO_URL` to the team's GitHub repo.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<team>/justiceflow-tz.git"

import os, sys
if REPO_URL:
    !git clone -q {REPO_URL} repo
    os.chdir("repo")
elif os.path.exists("justiceflow.zip"):
    !unzip -oq justiceflow.zip
!pip -q install ortools fairlearn
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.max_colwidth", 120)
from justiceflow import config as C, synthetic, pipeline, scheduler, ingest
from justiceflow.review import ReviewStore

## 2. Synthetic data, calibrated to official statistics
Cases are synthetic, fitted to published figures: NAO (2022) disposition targets, the Judiciary's 2020 backlog shares by court level, and 2025 filings by level. See `data/sources/SOURCES.md` for page references and the remaining assumptions.

In [ ]:
AS_OF = "2026-09-26"
data = synthetic.generate(n_cases=12000, seed=7, as_of=AS_OF)
print(len(data.cases), "cases |", len(data.hearings), "hearings |", len(data.magistrates), "magistrates")
print("Pending:", (data.cases.status == "Pending").sum())
print("Calibrated gap multipliers:", {k: round(v, 2) for k, v in data.gap_multiplier.items()})
data.cases.head()

## 3. Train the risk model behind the fairness gate (FR-1, FR-2, FR-4)

In [ ]:
engine = pipeline.train(data.cases, data.hearings, as_of=AS_OF)
print("Holdout:", engine.holdout_metrics)
print(engine.audit.summary())
engine.audit.table

## 4. Score pending cases with explanations and delay status (FR-3, FR-8)

In [ ]:
scored = pipeline.score(engine, data.cases, data.hearings, AS_OF)
print(scored.risk_category.value_counts().to_dict())
cols = ["case_id", "case_type", "court_level", "risk_probability", "risk_category", "delay_status", "driving_factors"]
scored[~scored.overdue].sort_values("risk_probability", ascending=False)[cols].head(10)

## 5. Propose next week's cause list with CP-SAT (FR-5) and compare with a greedy scheduler

In [ ]:
store = ReviewStore("review")
week_start = pd.Timestamp(AS_OF) + pd.offsets.Week(weekday=0)
cycle_id = f"W{week_start:%Y-%m-%d}"
proposal, info = pipeline.propose_week(engine, scored, data.magistrates, week_start, store, cycle_id)
print(info)
print("CP-SAT:", scheduler.compare(proposal, scored, data.magistrates))
print("Greedy:", scheduler.compare(scheduler.greedy(scored, data.magistrates, week_start), scored, data.magistrates))
proposal.head(15)

## 6. Registrar review (FR-6)
Nothing is scheduled until a registrar acts. Edits and removals need a reason and are logged.

In [ ]:
REGISTRAR = "REG-001"
top = proposal.sort_values("priority", ascending=False)
store.decide(cycle_id, top.case_id.iloc[0], "approve", REGISTRAR)
store.decide(cycle_id, top.case_id.iloc[1], "edit", REGISTRAR, day="Thu", reason="Advocate on circuit until Wednesday")
store.decide(cycle_id, top.case_id.iloc[2], "remove", REGISTRAR, reason="Accused not produced from remand")
# store.approve_all(cycle_id, REGISTRAR)   # explicit bulk approval, also logged
final = store.finalise(cycle_id, REGISTRAR)
print(len(final), "listings approved")
store.t["decisions"].tail()

## 7. Escalations (FR-7) — cases High for 3 consecutive cycles with no registrar action

In [ ]:
store.t["escalations"]  # fills once three cycles have run; `python -m justiceflow` replays three

## 8. Validate against the real TanzLII sample (FR-1)
Fill `data/tanzlii_sample_template.csv` by hand from tanzlii.org (no scraping; CC BY-NC 4.0, attribute TanzLII), upload it, then run.

In [ ]:
path = "data/tanzlii_sample.csv"
if os.path.exists(path):
    real = ingest.read_tanzlii_sample(path)
    print("TanzLII at-filing validation:", engine.model.evaluate(real))
else:
    print("No TanzLII sample uploaded yet.")

## 9. Export dashboard tables to Google Sheets (for Looker Studio)
Writes to JusticeFlow's own spreadsheet only — never to a Judiciary system.

In [ ]:
pipeline.export_dashboard(scored, store, engine, "dashboard")

EXPORT_TO_SHEETS = False
if EXPORT_TO_SHEETS:
    from google.colab import auth
    import gspread, google.auth
    auth.authenticate_user()
    gc = gspread.authorize(google.auth.default()[0])
    sh = gc.create("JusticeFlow TZ dashboard")
    for name in ["risk_scores", "proposals", "decisions", "escalations", "fairness_audit"]:
        df = pd.read_csv(f"dashboard/{name}.csv").fillna("").astype(str)
        ws = sh.add_worksheet(name, rows=len(df) + 1, cols=len(df.columns))
        ws.update([df.columns.tolist()] + df.values.tolist())
    print("Sheet:", sh.url)